# Phase 9 — Unstructured Magnitude Pruning

**Archived Kaggle research notebook.** Paths refer to the original Kaggle environment. Review path variables before execution. Some cleanup cells intentionally remove large temporary artifacts under `/kaggle/working`; run those cells only when the targets have been checked. Large datasets, model weights, adapters, and checkpoints are excluded from this GitHub repository.

In [ ]:
# ============================================================
# PHASE 9 — ENVIRONMENT CHECK
# ============================================================

import sys
import os
import torch
import transformers

print("=" * 80)
print("PHASE 9 — ENVIRONMENT")
print("=" * 80)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Transformers:", transformers.__version__)

print("\nGPUs:")
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    if "benchmark_2000.csv" in files:
        print("Found benchmark:", os.path.join(root, "benchmark_2000.csv"))

    if "metrics.csv" in files:
        print("Found metrics:", os.path.join(root, "metrics.csv"))

In [ ]:
def calculate_sparsity(model):
    total = 0
    zeros = 0

    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            w = module.weight.data
            total += w.numel()
            zeros += (w == 0).sum().item()

    return zeros / total

In [ ]:
# ============================================================
# PHASE 9 — LOAD PHASE 8 BENCHMARK + METRICS
# ============================================================

import os
import pandas as pd

PHASE8_DIR = "/kaggle/input/datasets/lucky10406/phase8-final-backup"

BENCHMARK_PATH = os.path.join(
    PHASE8_DIR,
    "benchmark_2000.csv"
)

METRICS_PATH = os.path.join(
    PHASE8_DIR,
    "metrics.csv"
)

benchmark_df = pd.read_csv(BENCHMARK_PATH)
phase8_metrics = pd.read_csv(METRICS_PATH)

print("Benchmark shape:", benchmark_df.shape)

print("\nPhase 8 models:")
print(phase8_metrics["model"].tolist())

print("\nPhase 8 metrics:")
display(phase8_metrics)

In [ ]:
# ============================================================
# CREATE PHASE 9 RESULTS AREA
# ============================================================

import shutil

PHASE9_DIR = "/kaggle/working/phase9-results"

os.makedirs(PHASE9_DIR, exist_ok=True)

shutil.copy2(
    METRICS_PATH,
    os.path.join(PHASE9_DIR, "metrics_phase8_base.csv")
)

benchmark_df.to_csv(
    os.path.join(PHASE9_DIR, "benchmark_2000.csv"),
    index=False
)

print("Phase 9 workspace ready:")
print(PHASE9_DIR)

print("\nFiles:")
for f in os.listdir(PHASE9_DIR):
    print("-", f)

In [ ]:
MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print(
    "Merged model already exists:",
    os.path.exists(MERGED_PATH)
)

In [ ]:
import os

ADAPTER_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "qwen2-5-3b-instruct-official/"
    "kaggle/working/qlora-adapter-final"
)

print("Adapter exists:", os.path.exists(ADAPTER_PATH))

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_PATH = "/kaggle/working/qwen2.5-3b-instruct"

if not os.path.exists(BASE_MODEL_PATH):
    snapshot_download(
        repo_id="Qwen/Qwen2.5-3B-Instruct",
        local_dir=BASE_MODEL_PATH
    )

print("Base model ready:", BASE_MODEL_PATH)

In [ ]:
!pip install -q --upgrade "torchao>=0.17.0"

In [ ]:
import torch
import transformers
import peft
import torchao

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("torchao:", torchao.__version__)

In [ ]:
import os

BASE_MODEL_PATH = "/kaggle/working/qwen2.5-3b-instruct"

print("Base model exists:", os.path.exists(BASE_MODEL_PATH))

In [ ]:
!pip install -q \
    "transformers==4.57.3" \
    "peft==0.19.1" \
    "bitsandbytes==0.50.1" \
    "accelerate==1.14.0"

In [ ]:
import torch
import transformers
import peft
import bitsandbytes as bnb
import accelerate
import torchao
import os

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("Accelerate:", accelerate.__version__)
print("torchao:", torchao.__version__)

BASE_MODEL_PATH = "/kaggle/working/qwen2.5-3b-instruct"
print("\nBase model exists:", os.path.exists(BASE_MODEL_PATH))

In [ ]:
# ============================================================
# PHASE 9 — MERGE FINAL QLORA ADAPTER INTO BASE MODEL
# ============================================================

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_PATH = "/kaggle/working/qwen2.5-3b-instruct"

ADAPTER_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "qwen2-5-3b-instruct-official/"
    "kaggle/working/qlora-adapter-final"
)

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("Base exists:", os.path.exists(BASE_MODEL_PATH))
print("Adapter exists:", os.path.exists(ADAPTER_PATH))

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    use_fast=True
)

# Load base model in FP16
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("\nBase model loaded.")

# Attach final LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("LoRA adapter loaded.")

# Merge adapter weights permanently into base model
merged_model = model.merge_and_unload()

print("LoRA merged.")

# Save merged FP16 model
merged_model.save_pretrained(
    MERGED_PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(MERGED_PATH)

print("\nMerged model saved to:")
print(MERGED_PATH)

In [ ]:
# ============================================================
# VERIFY MERGED MODEL
# ============================================================

import os

print("Merged directory exists:", os.path.exists(MERGED_PATH))

files = os.listdir(MERGED_PATH)

print("\nFiles:")
for f in sorted(files):
    print("-", f)

total_bytes = sum(
    os.path.getsize(os.path.join(MERGED_PATH, f))
    for f in files
    if os.path.isfile(os.path.join(MERGED_PATH, f))
)

print("\nApprox model directory size:")
print(round(total_bytes / (1024**3), 3), "GB")

In [ ]:
# ============================================================
# CLEAR MERGE MODEL FROM MEMORY
# ============================================================

import gc
import torch

del model
del base_model
del merged_model

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

print("GPU memory cleared.")

In [ ]:
# ============================================================
# PHASE 9 — 30% MAGNITUDE PRUNING
# ============================================================

import torch
import torch.nn.utils.prune as prune
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_PATH,
    use_fast=True
)

pruned_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Fresh merged model loaded.")

In [ ]:
import gc
import torch

try:
    del pruned_model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("Failed pruning model cleared.")

In [1]:
import os

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("Merged model exists:", os.path.exists(MERGED_PATH))

Merged model exists: True


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_PATH,
    use_fast=True
)

pruned_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map=None,
    low_cpu_mem_usage=True
)

pruned_model = pruned_model.cpu()

print("Model loaded on CPU.")
print("First parameter device:", next(pruned_model.parameters()).device)
print("First parameter dtype:", next(pruned_model.parameters()).dtype)

The tokenizer you are loading from '/kaggle/working/merged-finetuned-model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded on CPU.
First parameter device: cpu
First parameter dtype: torch.float16


In [3]:
import torch
import torch.nn as nn

@torch.no_grad()
def magnitude_prune_inplace(model, sparsity):
    total_pruned = 0
    total_weights = 0
    layers = 0

    for name, module in model.named_modules():

        if not isinstance(module, nn.Linear):
            continue

        weight = module.weight.data

        num_weights = weight.numel()
        num_to_prune = int(num_weights * sparsity)

        if num_to_prune == 0:
            continue

        # Work with one layer only
        flat_abs = weight.abs().reshape(-1)

        # Find indices of smallest-magnitude weights
        _, indices = torch.topk(
            flat_abs,
            k=num_to_prune,
            largest=False,
            sorted=False
        )

        # Permanently set those weights to zero
        weight.view(-1)[indices] = 0

        total_pruned += num_to_prune
        total_weights += num_weights
        layers += 1

        # Free temporary tensors immediately
        del flat_abs, indices

        if layers % 25 == 0:
            print(f"Processed {layers} linear layers...")

    print("\nPruning complete.")
    print("Linear layers processed:", layers)
    print("Weights targeted:", total_weights)
    print("Weights zeroed:", total_pruned)
    print(
        "Targeted sparsity:",
        round(total_pruned / total_weights * 100, 4),
        "%"
    )

    return model

In [4]:
pruned_model = magnitude_prune_inplace(
    pruned_model,
    sparsity=0.30
)

Processed 25 linear layers...
Processed 50 linear layers...
Processed 75 linear layers...
Processed 100 linear layers...
Processed 125 linear layers...
Processed 150 linear layers...
Processed 175 linear layers...
Processed 200 linear layers...
Processed 225 linear layers...
Processed 250 linear layers...

Pruning complete.
Linear layers processed: 253
Weights targeted: 3085697024
Weights zeroed: 925709042
Targeted sparsity: 30.0 %


In [5]:
@torch.no_grad()
def calculate_sparsity(model):
    total = 0
    zeros = 0
    layers = 0

    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            w = module.weight.data

            total += w.numel()
            zeros += (w == 0).sum().item()
            layers += 1

    return {
        "linear_layers": layers,
        "total_weights": total,
        "zero_weights": zeros,
        "sparsity": zeros / total
    }


info = calculate_sparsity(pruned_model)

print("Linear layers:", info["linear_layers"])
print("Total linear weights:", info["total_weights"])
print("Zero linear weights:", info["zero_weights"])
print(
    "Measured sparsity:",
    round(info["sparsity"] * 100, 4),
    "%"
)

Linear layers: 253
Total linear weights: 3085697024
Zero linear weights: 925709042
Measured sparsity: 30.0 %


In [6]:
PRUNED30_PATH = "/kaggle/working/pruned_30pct"

pruned_model.save_pretrained(
    PRUNED30_PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(PRUNED30_PATH)

print("30% pruned model saved:")
print(PRUNED30_PATH)

30% pruned model saved:
/kaggle/working/pruned_30pct


In [7]:
import os

def get_dir_size_gb(path):
    total = 0

    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            total += os.path.getsize(fp)

    return total / (1024 ** 3)


original_size = get_dir_size_gb(MERGED_PATH)
pruned_size = get_dir_size_gb(PRUNED30_PATH)

print("Original dense model:", round(original_size, 3), "GB")
print("30% pruned model:", round(pruned_size, 3), "GB")

Original dense model: 5.763 GB
30% pruned model: 5.763 GB


In [8]:
import gc

del pruned_model

gc.collect()
torch.cuda.empty_cache()

print("CPU pruning model cleared.")

CPU pruning model cleared.


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PRUNED30_PATH = "/kaggle/working/pruned_30pct"

tokenizer = AutoTokenizer.from_pretrained(PRUNED30_PATH)

pruned_model = AutoModelForCausalLM.from_pretrained(
    PRUNED30_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

pruned_model.eval()

print("30% pruned model loaded.")

The tokenizer you are loading from '/kaggle/working/pruned_30pct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

30% pruned model loaded.


In [10]:
LABELS = [
    "injured_or_dead_people",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "requests_or_urgent_needs",
    "missing_or_found_people",
    "other_relevant_information",
]

def format_prompt(tweet):
    return (
        "Classify the following disaster-related tweet into exactly "
        "one of these categories:\n"
        + "\n".join(LABELS)
        + "\n\n"
        f"Tweet: {tweet}\n\n"
        "Answer:"
    )

In [11]:
def predict(tweet, model, tokenizer):

    prompt = format_prompt(tweet)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False
    )

    inputs.pop("token_type_ids", None)

    device = next(model.parameters()).device
    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # Match exact allowed class
    for label in LABELS:
        if label.lower() in text.lower():
            return label

    return "unmatched"

In [13]:
import pandas as pd

BENCHMARK_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv"
)

benchmark_df = pd.read_csv(BENCHMARK_PATH)

print("Benchmark loaded:", benchmark_df.shape)
print(benchmark_df.columns.tolist())

Benchmark loaded: (2000, 3)
['tweet_text', 'class_label', 'text_clean']


In [14]:
sample_df = benchmark_df.iloc[:5]

for i, row in sample_df.iterrows():

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    print("=" * 80)
    print("Tweet:", row["text_clean"])
    print("True :", row["class_label"])
    print("Pred :", pred)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tweet: Hey @MarcAnthony aid arrived days ago! Its an island. No neighboring states can help. Ancient electric grid.
True : rescue_volunteering_or_donation_effort
Pred : rescue_volunteering_or_donation_effort
Tweet: ὏8- Besiktas fans showing their support to the people in Greece who were affected by the wildfires. #Πυρκαγια
True : sympathy_and_support
Pred : sympathy_and_support
Tweet: Recommendations: (v)Guarantee the provision of access to clean water and food for all those affected by the floods; #CycloneIdai
True : other_relevant_information
Pred : requests_or_urgent_needs
Tweet: At 12:15 PM EDT, Plantersville [Lunenburg Co, VA] PUBLIC reports FLASH FLOOD. PLANTERVSILLE RD ROAD CLOSED DUE TO FLASH FLOODING. NUMEROUS SECONDARY ROADS IN THE AREA ALSO FLOODED AND IMPASSIBLE.
True : caution_and_advice
Pred : not_humanitarian
Tweet: BAHAMAS: Thoughts and prayers with the wonderful people of the Abacos in the Bahamas as they face the catastrophic fury of Dorian.
True : sympathy_and_suppor

In [15]:
# ============================================================
# PHASE 9 — 30% PRUNED MODEL
# FULL 2000-EXAMPLE QUALITY BENCHMARK
# ============================================================

import time
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

predictions = []
raw_times = []

print("Starting 30% pruning benchmark...")
print("Examples:", len(benchmark_df))

start_total = time.perf_counter()

for idx, row in benchmark_df.iterrows():

    t0 = time.perf_counter()

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    t1 = time.perf_counter()

    predictions.append(pred)
    raw_times.append(t1 - t0)

    if (idx + 1) % 100 == 0:
        elapsed = time.perf_counter() - start_total

        print(
            f"{idx + 1}/{len(benchmark_df)} completed "
            f"| elapsed: {elapsed/60:.1f} min"
        )

total_time = time.perf_counter() - start_total

print("\nBenchmark complete.")
print("Total inference time:", round(total_time, 2), "seconds")

Starting 30% pruning benchmark...
Examples: 2000
100/2000 completed | elapsed: 1.7 min
200/2000 completed | elapsed: 3.4 min
300/2000 completed | elapsed: 5.1 min
400/2000 completed | elapsed: 6.8 min
500/2000 completed | elapsed: 8.5 min
600/2000 completed | elapsed: 10.2 min
700/2000 completed | elapsed: 11.9 min
800/2000 completed | elapsed: 13.6 min
900/2000 completed | elapsed: 15.2 min
1000/2000 completed | elapsed: 16.9 min
1100/2000 completed | elapsed: 18.6 min
1200/2000 completed | elapsed: 20.3 min
1300/2000 completed | elapsed: 22.0 min
1400/2000 completed | elapsed: 23.7 min
1500/2000 completed | elapsed: 25.3 min
1600/2000 completed | elapsed: 27.0 min
1700/2000 completed | elapsed: 28.7 min
1800/2000 completed | elapsed: 30.4 min
1900/2000 completed | elapsed: 32.1 min
2000/2000 completed | elapsed: 33.7 min

Benchmark complete.
Total inference time: 2024.7 seconds


In [16]:
# ============================================================
# CALCULATE QUALITY METRICS
# ============================================================

y_true = benchmark_df["class_label"].tolist()
y_pred = predictions

accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="weighted",
    zero_division=0
)

samples_per_second = len(benchmark_df) / total_time

unmatched_count = sum(
    pred == "unmatched"
    for pred in y_pred
)

results_30 = {
    "model": "pruned_30pct",
    "sparsity": 0.30,
    "n_samples": len(benchmark_df),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "total_inference_time_s": total_time,
    "samples_per_second": samples_per_second,
    "unmatched_count": unmatched_count
}

print("=" * 70)
print("30% PRUNING RESULTS")
print("=" * 70)

for key, value in results_30.items():
    print(f"{key}: {value}")

30% PRUNING RESULTS
model: pruned_30pct
sparsity: 0.3
n_samples: 2000
accuracy: 0.6395
macro_precision: 0.7673761472315012
macro_recall: 0.6483001095603204
macro_f1: 0.6333022599482043
weighted_f1: 0.6318104752816464
total_inference_time_s: 2024.696894444
samples_per_second: 0.9878021769521299
unmatched_count: 6


In [17]:
# ============================================================
# SAVE 30% RESULTS + PREDICTIONS
# ============================================================

import os

PHASE9_DIR = "/kaggle/working/phase9-results"

os.makedirs(PHASE9_DIR, exist_ok=True)

with open(
    os.path.join(PHASE9_DIR, "pruned30_2000_results.json"),
    "w"
) as f:
    json.dump(results_30, f, indent=4)


pred_df = benchmark_df.copy()

pred_df["prediction"] = predictions
pred_df["correct"] = (
    pred_df["class_label"] == pred_df["prediction"]
)
pred_df["inference_time_s"] = raw_times

pred_df.to_csv(
    os.path.join(
        PHASE9_DIR,
        "pruned30_2000_predictions.csv"
    ),
    index=False
)

print("Saved Phase 9 30% quality results.")

Saved Phase 9 30% quality results.


In [18]:
# ============================================================
# SAVE CLASSIFICATION REPORT + CONFUSION MATRIX
# ============================================================

report = classification_report(
    y_true,
    y_pred,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0
)

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned30_classification_report.txt"
    ),
    "w"
) as f:
    f.write(report)


cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

cm_df.to_csv(
    os.path.join(
        PHASE9_DIR,
        "pruned30_confusion_matrix.csv"
    )
)

print(report)

                                        precision    recall  f1-score   support

                injured_or_dead_people       0.93      0.89      0.91       191
rescue_volunteering_or_donation_effort       0.87      0.80      0.83       556
                  sympathy_and_support       0.97      0.74      0.84       234
     infrastructure_and_utility_damage       0.80      0.79      0.79       214
                      not_humanitarian       0.24      0.95      0.38       165
                    caution_and_advice       1.00      0.06      0.12       141
      displaced_people_and_evacuations       0.84      0.87      0.85       105
              requests_or_urgent_needs       0.69      0.40      0.50        68
               missing_or_found_people       1.00      0.89      0.94         9
            other_relevant_information       0.33      0.10      0.15       317

                             micro avg       0.64      0.64      0.64      2000
                             macro avg

In [19]:
# ============================================================
# PHASE 9 — 30% PRUNING LATENCY + THROUGHPUT
# Same methodology as Phase 8
# ============================================================

import time
import json
import os
import numpy as np
import torch

PHASE9_DIR = "/kaggle/working/phase9-results"

LATENCY_TWEET = benchmark_df["text_clean"].iloc[0]

# ---------- Warmup ----------
print("Running 5 warmup inferences...")

for _ in range(5):
    _ = predict(
        LATENCY_TWEET,
        pruned_model,
        tokenizer
    )

# ---------- Measured trials ----------
times = []

print("Running 50 latency trials...")

for i in range(50):

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    _ = predict(
        LATENCY_TWEET,
        pruned_model,
        tokenizer
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end = time.perf_counter()

    times.append(end - start)

latency_30 = {
    "model": "pruned_30pct",
    "warmup_runs": 5,
    "trials": 50,
    "mean_latency_s": float(np.mean(times)),
    "median_latency_s": float(np.median(times)),
    "std_latency_s": float(np.std(times)),
    "min_latency_s": float(np.min(times)),
    "max_latency_s": float(np.max(times)),
    "throughput_samples_per_s": float(1 / np.mean(times))
}

print("\n30% PRUNING — LATENCY RESULTS")
print("=" * 60)

for k, v in latency_30.items():
    print(k, ":", v)

with open(
    os.path.join(PHASE9_DIR, "pruned30_runtime.json"),
    "w"
) as f:
    json.dump(latency_30, f, indent=4)

Running 5 warmup inferences...
Running 50 latency trials...

30% PRUNING — LATENCY RESULTS
model : pruned_30pct
warmup_runs : 5
trials : 50
mean_latency_s : 1.0051097394800126
median_latency_s : 1.001759561999961
std_latency_s : 0.015050742984803715
min_latency_s : 0.9801469510002789
max_latency_s : 1.0468525329997647
throughput_samples_per_s : 0.9949162372233542


In [20]:
# ============================================================
# 30% PRUNING — GPU MEMORY
# ============================================================

import gc

gc.collect()
torch.cuda.empty_cache()

# Reset peak statistics for every GPU
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

# One normal inference
_ = predict(
    LATENCY_TWEET,
    pruned_model,
    tokenizer
)

memory_30 = {
    "model": "pruned_30pct",
    "gpus": {}
}

for i in range(torch.cuda.device_count()):

    memory_30["gpus"][f"gpu_{i}"] = {
        "allocated_gb":
            torch.cuda.memory_allocated(i) / (1024**3),

        "reserved_gb":
            torch.cuda.memory_reserved(i) / (1024**3),

        "max_allocated_gb":
            torch.cuda.max_memory_allocated(i) / (1024**3),

        "max_reserved_gb":
            torch.cuda.max_memory_reserved(i) / (1024**3)
    }


print("\n30% PRUNING — GPU MEMORY")
print("=" * 60)

for gpu, vals in memory_30["gpus"].items():

    print("\n", gpu)

    for k, v in vals.items():
        print(f"{k}: {v:.4f}")


with open(
    os.path.join(PHASE9_DIR, "pruned30_memory.json"),
    "w"
) as f:
    json.dump(memory_30, f, indent=4)


30% PRUNING — GPU MEMORY

 gpu_0
allocated_gb: 2.8856
reserved_gb: 2.9180
max_allocated_gb: 2.8954
max_reserved_gb: 2.9180

 gpu_1
allocated_gb: 2.8803
reserved_gb: 2.9219
max_allocated_gb: 2.8901
max_reserved_gb: 2.9219


In [21]:
ENERGY_SAMPLE_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/energy_sample_100.csv"
)

print("Energy sample exists:",
      os.path.exists(ENERGY_SAMPLE_PATH))

Energy sample exists: True


In [22]:
import pandas as pd

energy_df = pd.read_csv(ENERGY_SAMPLE_PATH)

print("Energy sample shape:", energy_df.shape)
print(energy_df.columns.tolist())

Energy sample shape: (100, 3)
['tweet_text', 'class_label', 'text_clean']


In [23]:
try:
    from codecarbon import EmissionsTracker
    print("CodeCarbon ready.")
except ImportError:
    !pip install -q codecarbon
    from codecarbon import EmissionsTracker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.9/396.9 kB 6.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 66.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 99.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.3.0 requires cryptography<49,>=42, but you have cryptography 50.0.1 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 50.0.1 which is incompatible.
pyopenssl 24.2.1 requires cryptogra

In [24]:
# ============================================================
# 30% PRUNING — ENERGY / CARBON
# Exact Phase 8 100-example workload
# ============================================================

from codecarbon import EmissionsTracker
import time
import os
import json

ENERGY_LOG_DIR = os.path.join(
    PHASE9_DIR,
    "energy_logs",
    "pruned30"
)

os.makedirs(
    ENERGY_LOG_DIR,
    exist_ok=True
)

tracker = EmissionsTracker(
    output_dir=ENERGY_LOG_DIR,
    output_file="emissions.csv",
    log_level="error"
)

print("Starting CodeCarbon measurement...")

tracker.start()

energy_start = time.perf_counter()

energy_predictions_30 = []

for i, row in energy_df.iterrows():

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    energy_predictions_30.append(pred)

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/100")

runtime_energy = (
    time.perf_counter() - energy_start
)

emissions_kg = tracker.stop()

print("\nEnergy workload complete.")
print("Runtime:", runtime_energy, "seconds")
print("Emissions:", emissions_kg, "kg CO2e")

[codecarbon WARNING @ 09:39:56] Multiple instances of codecarbon are allowed to run at the same time.


Starting CodeCarbon measurement...
20/100
40/100
60/100
80/100
100/100

Energy workload complete.
Runtime: 101.98472122899966 seconds
Emissions: 0.0013464231782782954 kg CO2e


In [25]:
cc_path = os.path.join(
    ENERGY_LOG_DIR,
    "emissions.csv"
)

cc_df = pd.read_csv(cc_path)

display(
    cc_df.tail(1).T
)

,0
timestamp,2026-08-29T09:41:43
project_name,codecarbon
run_id,91355879-2cab-45e9-ad6f-4e419c363a01
experiment_id,5b0fa12a-3dd7-45bb-9766-cc326314d9f1
duration,105.178048
emissions,0.001346
emissions_rate,0.000013
cpu_power,1.942984
gpu_power,85.946794
ram_power,20.0


In [26]:
last = cc_df.iloc[-1]

energy_30 = {
    "model": "pruned_30pct",
    "n_samples": 100,
    "runtime_s": float(runtime_energy),
    "energy_kwh": float(last["energy_consumed"]),
    "emissions_kg_co2": float(last["emissions"])
}

print("\n30% PRUNING — ENERGY RESULTS")
print("=" * 60)

for k, v in energy_30.items():
    print(k, ":", v)


with open(
    os.path.join(
        PHASE9_DIR,
        "pruned30_energy.json"
    ),
    "w"
) as f:

    json.dump(
        energy_30,
        f,
        indent=4
    )


30% PRUNING — ENERGY RESULTS
model : pruned_30pct
n_samples : 100
runtime_s : 101.98472122899966
energy_kwh : 0.002974726862353
emissions_kg_co2 : 0.0013464231782782


In [27]:
with open(
    os.path.join(
        PHASE9_DIR,
        "pruned30_2000_results.json"
    )
) as f:

    exact30 = json.load(f)


print("EXACT 30% RESULTS")
print("=" * 60)

for k, v in exact30.items():
    print(k, ":", v)

EXACT 30% RESULTS
model : pruned_30pct
sparsity : 0.3
n_samples : 2000
accuracy : 0.6395
macro_precision : 0.7673761472315012
macro_recall : 0.6483001095603204
macro_f1 : 0.6333022599482043
weighted_f1 : 0.6318104752816464
total_inference_time_s : 2024.696894444
samples_per_second : 0.9878021769521299
unmatched_count : 6


In [28]:
import gc
import torch

del pruned_model

gc.collect()
torch.cuda.empty_cache()

print("30% model cleared from GPU.")

30% model cleared from GPU.


In [29]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_PATH,
    use_fast=True
)

pruned_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map=None,
    low_cpu_mem_usage=True
)

pruned_model = pruned_model.cpu()

print("Fresh original model loaded on CPU.")
print("Device:", next(pruned_model.parameters()).device)

The tokenizer you are loading from '/kaggle/working/merged-finetuned-model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fresh original model loaded on CPU.
Device: cpu


In [30]:
import torch.nn as nn

@torch.no_grad()
def magnitude_prune_inplace(model, sparsity):
    total_pruned = 0
    total_weights = 0
    layers = 0

    for name, module in model.named_modules():

        if not isinstance(module, nn.Linear):
            continue

        weight = module.weight.data

        num_weights = weight.numel()
        num_to_prune = int(num_weights * sparsity)

        if num_to_prune == 0:
            continue

        flat_abs = weight.abs().reshape(-1)

        _, indices = torch.topk(
            flat_abs,
            k=num_to_prune,
            largest=False,
            sorted=False
        )

        weight.view(-1)[indices] = 0

        total_pruned += num_to_prune
        total_weights += num_weights
        layers += 1

        del flat_abs, indices

        if layers % 25 == 0:
            print(f"Processed {layers} linear layers...")

    print("\nPruning complete.")
    print("Linear layers:", layers)
    print("Weights zeroed:", total_pruned)
    print(
        "Targeted sparsity:",
        round(total_pruned / total_weights * 100, 4),
        "%"
    )

    return model

In [31]:
pruned_model = magnitude_prune_inplace(
    pruned_model,
    sparsity=0.50
)

Processed 25 linear layers...
Processed 50 linear layers...
Processed 75 linear layers...
Processed 100 linear layers...
Processed 125 linear layers...
Processed 150 linear layers...
Processed 175 linear layers...
Processed 200 linear layers...
Processed 225 linear layers...
Processed 250 linear layers...

Pruning complete.
Linear layers: 253
Weights zeroed: 1542848512
Targeted sparsity: 50.0 %


In [32]:
@torch.no_grad()
def calculate_sparsity(model):
    total = 0
    zeros = 0

    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            w = module.weight.data

            total += w.numel()
            zeros += (w == 0).sum().item()

    return zeros / total


actual_sparsity = calculate_sparsity(pruned_model)

print(
    "Measured sparsity:",
    round(actual_sparsity * 100, 4),
    "%"
)

Measured sparsity: 50.0 %


In [34]:
import os
import shutil

PRUNED30_PATH = "/kaggle/working/pruned_30pct"
PRUNED50_PATH = "/kaggle/working/pruned_50pct"

# Remove partially-written 50% checkpoint
if os.path.exists(PRUNED50_PATH):
    shutil.rmtree(PRUNED50_PATH)
    print("Removed partial 50% folder.")

# 30% benchmarking is already complete, so its huge checkpoint
# is no longer needed.
if os.path.exists(PRUNED30_PATH):
    shutil.rmtree(PRUNED30_PATH)
    print("Removed finished 30% checkpoint.")

print("Disk space reclaimed.")

Removed partial 50% folder.
Removed finished 30% checkpoint.
Disk space reclaimed.


In [35]:
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")

print("Total:", round(total / 1024**3, 2), "GB")
print("Used :", round(used / 1024**3, 2), "GB")
print("Free :", round(free / 1024**3, 2), "GB")

Total: 19.52 GB
Used : 11.52 GB
Free : 7.98 GB


In [36]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

pruned_model = pruned_model.to("cuda:0")

pruned_model.eval()

print("50% pruned model moved to GPU.")
print("Device:", next(pruned_model.parameters()).device)

50% pruned model moved to GPU.
Device: cuda:0


In [37]:
# ============================================================
# SAVE 50% MODEL AFTER DISK CLEANUP
# ============================================================

import os
import gc
import torch

PRUNED50_PATH = "/kaggle/working/pruned_50pct"

# Move back to CPU for clean saving
pruned_model = pruned_model.cpu()

gc.collect()
torch.cuda.empty_cache()

pruned_model.save_pretrained(
    PRUNED50_PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(PRUNED50_PATH)

print("50% checkpoint saved.")

50% checkpoint saved.


In [38]:
def get_dir_size_gb(path):
    total = 0

    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            total += os.path.getsize(fp)

    return total / (1024**3)


print(
    "50% pruned checkpoint:",
    round(get_dir_size_gb(PRUNED50_PATH), 3),
    "GB"
)

50% pruned checkpoint: 5.763 GB


In [39]:
del pruned_model

gc.collect()
torch.cuda.empty_cache()

In [40]:
from transformers import AutoModelForCausalLM

pruned_model = AutoModelForCausalLM.from_pretrained(
    PRUNED50_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

pruned_model.eval()

print("50% model loaded with device_map='auto'.")

print("\nDevice map:")
print(pruned_model.hf_device_map)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

50% model loaded with device_map='auto'.

Device map:
{'': 0}


In [41]:
sample_df = benchmark_df.iloc[:5]

for i, row in sample_df.iterrows():

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    print("=" * 80)
    print("Tweet:", row["text_clean"])
    print("True :", row["class_label"])
    print("Pred :", pred)

Tweet: Hey @MarcAnthony aid arrived days ago! Its an island. No neighboring states can help. Ancient electric grid.
True : rescue_volunteering_or_donation_effort
Pred : unmatched
Tweet: ὏8- Besiktas fans showing their support to the people in Greece who were affected by the wildfires. #Πυρκαγια
True : sympathy_and_support
Pred : unmatched
Tweet: Recommendations: (v)Guarantee the provision of access to clean water and food for all those affected by the floods; #CycloneIdai
True : other_relevant_information
Pred : unmatched
Tweet: At 12:15 PM EDT, Plantersville [Lunenburg Co, VA] PUBLIC reports FLASH FLOOD. PLANTERVSILLE RD ROAD CLOSED DUE TO FLASH FLOODING. NUMEROUS SECONDARY ROADS IN THE AREA ALSO FLOODED AND IMPASSIBLE.
True : caution_and_advice
Pred : caution_and_advice
Tweet: BAHAMAS: Thoughts and prayers with the wonderful people of the Abacos in the Bahamas as they face the catastrophic fury of Dorian.
True : sympathy_and_support
Pred : unmatched


In [42]:
# ============================================================
# PHASE 9 — 50% PRUNING
# FULL 2000-EXAMPLE QUALITY BENCHMARK
# ============================================================

import time
import json
import os

predictions_50 = []
raw_times_50 = []

print("Starting 50% pruning benchmark...")
print("Examples:", len(benchmark_df))

start_total = time.perf_counter()

for idx, row in benchmark_df.iterrows():

    t0 = time.perf_counter()

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    t1 = time.perf_counter()

    predictions_50.append(pred)
    raw_times_50.append(t1 - t0)

    if (idx + 1) % 100 == 0:
        elapsed = time.perf_counter() - start_total
        print(
            f"{idx + 1}/{len(benchmark_df)} completed "
            f"| elapsed: {elapsed/60:.1f} min"
        )

total_time_50 = time.perf_counter() - start_total

print("\nBenchmark complete.")
print("Total inference time:", round(total_time_50, 2), "seconds")

Starting 50% pruning benchmark...
Examples: 2000
100/2000 completed | elapsed: 1.3 min
200/2000 completed | elapsed: 2.6 min
300/2000 completed | elapsed: 3.9 min
400/2000 completed | elapsed: 5.2 min
500/2000 completed | elapsed: 6.4 min
600/2000 completed | elapsed: 7.7 min
700/2000 completed | elapsed: 9.0 min
800/2000 completed | elapsed: 10.3 min
900/2000 completed | elapsed: 11.6 min
1000/2000 completed | elapsed: 12.9 min
1100/2000 completed | elapsed: 14.2 min
1200/2000 completed | elapsed: 15.4 min
1300/2000 completed | elapsed: 16.7 min
1400/2000 completed | elapsed: 18.0 min
1500/2000 completed | elapsed: 19.3 min
1600/2000 completed | elapsed: 20.6 min
1700/2000 completed | elapsed: 21.9 min
1800/2000 completed | elapsed: 23.1 min
1900/2000 completed | elapsed: 24.4 min
2000/2000 completed | elapsed: 25.7 min

Benchmark complete.
Total inference time: 1540.87 seconds


In [43]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

y_true = benchmark_df["class_label"].tolist()
y_pred_50 = predictions_50

results_50 = {
    "model": "pruned_50pct",
    "sparsity": 0.50,
    "n_samples": len(benchmark_df),

    "accuracy": accuracy_score(
        y_true,
        y_pred_50
    ),

    "macro_precision": precision_score(
        y_true,
        y_pred_50,
        labels=LABELS,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true,
        y_pred_50,
        labels=LABELS,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true,
        y_pred_50,
        labels=LABELS,
        average="macro",
        zero_division=0
    ),

    "weighted_f1": f1_score(
        y_true,
        y_pred_50,
        labels=LABELS,
        average="weighted",
        zero_division=0
    ),

    "total_inference_time_s":
        total_time_50,

    "samples_per_second":
        len(benchmark_df) / total_time_50,

    "unmatched_count":
        sum(
            p == "unmatched"
            for p in y_pred_50
        )
}

print("=" * 70)
print("50% PRUNING RESULTS")
print("=" * 70)

for k, v in results_50.items():
    print(k, ":", v)

50% PRUNING RESULTS
model : pruned_50pct
sparsity : 0.5
n_samples : 2000
accuracy : 0.0005
macro_precision : 0.05
macro_recall : 0.0007092198581560283
macro_f1 : 0.0013986013986013986
weighted_f1 : 0.000986013986013986
total_inference_time_s : 1540.8703148759996
samples_per_second : 1.2979677658083435
unmatched_count : 1992


In [44]:
PHASE9_DIR = "/kaggle/working/phase9-results"

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned50_2000_results.json"
    ),
    "w"
) as f:
    json.dump(results_50, f, indent=4)


pred_df_50 = benchmark_df.copy()

pred_df_50["prediction"] = predictions_50
pred_df_50["correct"] = (
    pred_df_50["class_label"]
    == pred_df_50["prediction"]
)

pred_df_50["inference_time_s"] = raw_times_50

pred_df_50.to_csv(
    os.path.join(
        PHASE9_DIR,
        "pruned50_2000_predictions.csv"
    ),
    index=False
)

In [45]:
report_50 = classification_report(
    y_true,
    y_pred_50,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0
)

print(report_50)

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned50_classification_report.txt"
    ),
    "w"
) as f:
    f.write(report_50)


cm_50 = confusion_matrix(
    y_true,
    y_pred_50,
    labels=LABELS
)

pd.DataFrame(
    cm_50,
    index=LABELS,
    columns=LABELS
).to_csv(
    os.path.join(
        PHASE9_DIR,
        "pruned50_confusion_matrix.csv"
    )
)

                                        precision    recall  f1-score   support

                injured_or_dead_people       0.00      0.00      0.00       191
rescue_volunteering_or_donation_effort       0.00      0.00      0.00       556
                  sympathy_and_support       0.00      0.00      0.00       234
     infrastructure_and_utility_damage       0.00      0.00      0.00       214
                      not_humanitarian       0.00      0.00      0.00       165
                    caution_and_advice       0.50      0.01      0.01       141
      displaced_people_and_evacuations       0.00      0.00      0.00       105
              requests_or_urgent_needs       0.00      0.00      0.00        68
               missing_or_found_people       0.00      0.00      0.00         9
            other_relevant_information       0.00      0.00      0.00       317

                             micro avg       0.12      0.00      0.00      2000
                             macro avg

In [46]:
print("EXACT 50% RESULTS")
print("=" * 60)

for k, v in results_50.items():
    print(k, ":", v)

EXACT 50% RESULTS
model : pruned_50pct
sparsity : 0.5
n_samples : 2000
accuracy : 0.0005
macro_precision : 0.05
macro_recall : 0.0007092198581560283
macro_f1 : 0.0013986013986013986
weighted_f1 : 0.000986013986013986
total_inference_time_s : 1540.8703148759996
samples_per_second : 1.2979677658083435
unmatched_count : 1992


In [47]:
# ============================================================
# 50% PRUNING — LATENCY + THROUGHPUT
# ============================================================

import time
import numpy as np
import torch
import json
import os

LATENCY_TWEET = benchmark_df["text_clean"].iloc[0]

for _ in range(5):
    _ = predict(LATENCY_TWEET, pruned_model, tokenizer)

times_50 = []

for _ in range(50):

    torch.cuda.synchronize()

    start = time.perf_counter()

    _ = predict(
        LATENCY_TWEET,
        pruned_model,
        tokenizer
    )

    torch.cuda.synchronize()

    times_50.append(
        time.perf_counter() - start
    )


latency_50 = {
    "model": "pruned_50pct",
    "warmup_runs": 5,
    "trials": 50,
    "mean_latency_s": float(np.mean(times_50)),
    "median_latency_s": float(np.median(times_50)),
    "std_latency_s": float(np.std(times_50)),
    "min_latency_s": float(np.min(times_50)),
    "max_latency_s": float(np.max(times_50)),
    "throughput_samples_per_s":
        float(1 / np.mean(times_50))
}

print("50% PRUNING — LATENCY RESULTS")
print("=" * 60)

for k, v in latency_50.items():
    print(k, ":", v)

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned50_runtime.json"
    ),
    "w"
) as f:
    json.dump(latency_50, f, indent=4)

50% PRUNING — LATENCY RESULTS
model : pruned_50pct
warmup_runs : 5
trials : 50
mean_latency_s : 0.7544760617600695
median_latency_s : 0.7536016564999954
std_latency_s : 0.010883740523480916
min_latency_s : 0.7397563720005564
max_latency_s : 0.7924685680000039
throughput_samples_per_s : 1.3254230991334082


In [48]:
# ============================================================
# 50% PRUNING — GPU MEMORY
# ============================================================

import gc

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

_ = predict(
    LATENCY_TWEET,
    pruned_model,
    tokenizer
)

memory_50 = {
    "model": "pruned_50pct",
    "gpus": {}
}

for i in range(torch.cuda.device_count()):

    memory_50["gpus"][f"gpu_{i}"] = {
        "allocated_gb":
            torch.cuda.memory_allocated(i) / 1024**3,

        "reserved_gb":
            torch.cuda.memory_reserved(i) / 1024**3,

        "max_allocated_gb":
            torch.cuda.max_memory_allocated(i) / 1024**3,

        "max_reserved_gb":
            torch.cuda.max_memory_reserved(i) / 1024**3
    }

print("50% PRUNING — GPU MEMORY")
print("=" * 60)

for gpu, vals in memory_50["gpus"].items():

    print("\n", gpu)

    for k, v in vals.items():
        print(f"{k}: {v:.4f}")


with open(
    os.path.join(
        PHASE9_DIR,
        "pruned50_memory.json"
    ),
    "w"
) as f:
    json.dump(memory_50, f, indent=4)

50% PRUNING — GPU MEMORY

 gpu_0
allocated_gb: 5.7569
reserved_gb: 5.8398
max_allocated_gb: 5.7688
max_reserved_gb: 5.8398

 gpu_1
allocated_gb: 0.0089
reserved_gb: 2.8730
max_allocated_gb: 0.0089
max_reserved_gb: 2.8730


In [49]:
# ============================================================
# 50% PRUNING — ENERGY + CO2
# ============================================================

from codecarbon import EmissionsTracker
import pandas as pd
import time

ENERGY_SAMPLE_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/energy_sample_100.csv"
)

energy_df = pd.read_csv(ENERGY_SAMPLE_PATH)

ENERGY50_DIR = os.path.join(
    PHASE9_DIR,
    "energy_logs",
    "pruned50"
)

os.makedirs(ENERGY50_DIR, exist_ok=True)

tracker = EmissionsTracker(
    output_dir=ENERGY50_DIR,
    output_file="emissions.csv",
    log_level="error"
)

tracker.start()

start = time.perf_counter()

energy_predictions_50 = []

for i, row in energy_df.iterrows():

    energy_predictions_50.append(
        predict(
            row["text_clean"],
            pruned_model,
            tokenizer
        )
    )

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/100")

runtime_50_energy = (
    time.perf_counter() - start
)

tracker.stop()

cc50 = pd.read_csv(
    os.path.join(
        ENERGY50_DIR,
        "emissions.csv"
    )
)

last = cc50.iloc[-1]

energy_50 = {
    "model": "pruned_50pct",
    "n_samples": 100,
    "runtime_s": float(runtime_50_energy),
    "energy_kwh":
        float(last["energy_consumed"]),
    "emissions_kg_co2":
        float(last["emissions"])
}

print("\n50% PRUNING — ENERGY RESULTS")
print("=" * 60)

for k, v in energy_50.items():
    print(k, ":", v)

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned50_energy.json"
    ),
    "w"
) as f:
    json.dump(energy_50, f, indent=4)

20/100
40/100
60/100
80/100
100/100

50% PRUNING — ENERGY RESULTS
model : pruned_50pct
n_samples : 100
runtime_s : 76.9602395740003
energy_kwh : 0.0025160002320851
emissions_kg_co2 : 0.0011387939753075


In [50]:
import gc
import os
import shutil
import torch

try:
    del pruned_model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

PRUNED50_PATH = "/kaggle/working/pruned_50pct"

if os.path.exists(PRUNED50_PATH):
    shutil.rmtree(PRUNED50_PATH)

print("50% model cleared.")

50% model cleared.


In [51]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_PATH,
    use_fast=True
)

pruned_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map=None,
    low_cpu_mem_usage=True
).cpu()

print("Fresh merged model loaded on CPU.")

The tokenizer you are loading from '/kaggle/working/merged-finetuned-model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fresh merged model loaded on CPU.


In [52]:
pruned_model = magnitude_prune_inplace(
    pruned_model,
    sparsity=0.70
)

Processed 25 linear layers...
Processed 50 linear layers...
Processed 75 linear layers...
Processed 100 linear layers...
Processed 125 linear layers...
Processed 150 linear layers...
Processed 175 linear layers...
Processed 200 linear layers...
Processed 225 linear layers...
Processed 250 linear layers...

Pruning complete.
Linear layers: 253
Weights zeroed: 2159987729
Targeted sparsity: 70.0 %


In [53]:
actual_sparsity_70 = calculate_sparsity(pruned_model)

print(
    "Measured sparsity:",
    round(actual_sparsity_70 * 100, 4),
    "%"
)

Measured sparsity: 70.0 %


In [54]:
gc.collect()
torch.cuda.empty_cache()

pruned_model = pruned_model.to("cuda:0")
pruned_model.eval()

print("70% model ready.")

70% model ready.


In [55]:
sample_df = benchmark_df.iloc[:5]

for _, row in sample_df.iterrows():

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    print("=" * 80)
    print("Tweet:", row["text_clean"])
    print("True :", row["class_label"])
    print("Pred :", pred)

Tweet: Hey @MarcAnthony aid arrived days ago! Its an island. No neighboring states can help. Ancient electric grid.
True : rescue_volunteering_or_donation_effort
Pred : unmatched
Tweet: ὏8- Besiktas fans showing their support to the people in Greece who were affected by the wildfires. #Πυρκαγια
True : sympathy_and_support
Pred : unmatched
Tweet: Recommendations: (v)Guarantee the provision of access to clean water and food for all those affected by the floods; #CycloneIdai
True : other_relevant_information
Pred : unmatched
Tweet: At 12:15 PM EDT, Plantersville [Lunenburg Co, VA] PUBLIC reports FLASH FLOOD. PLANTERVSILLE RD ROAD CLOSED DUE TO FLASH FLOODING. NUMEROUS SECONDARY ROADS IN THE AREA ALSO FLOODED AND IMPASSIBLE.
True : caution_and_advice
Pred : unmatched
Tweet: BAHAMAS: Thoughts and prayers with the wonderful people of the Abacos in the Bahamas as they face the catastrophic fury of Dorian.
True : sympathy_and_support
Pred : unmatched


In [56]:
# ============================================================
# 70% PRUNING — FULL 2000-EXAMPLE QUALITY BENCHMARK
# ============================================================

import time
import json
import os

predictions_70 = []
raw_times_70 = []

print("Starting 70% pruning benchmark...")

start_total = time.perf_counter()

for idx, row in benchmark_df.iterrows():

    t0 = time.perf_counter()

    pred = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    raw_times_70.append(
        time.perf_counter() - t0
    )

    predictions_70.append(pred)

    if (idx + 1) % 100 == 0:
        print(
            f"{idx + 1}/2000 "
            f"| elapsed: "
            f"{(time.perf_counter()-start_total)/60:.1f} min"
        )

total_time_70 = time.perf_counter() - start_total

Starting 70% pruning benchmark...
100/2000 | elapsed: 1.3 min
200/2000 | elapsed: 2.5 min
300/2000 | elapsed: 3.8 min
400/2000 | elapsed: 5.0 min
500/2000 | elapsed: 6.2 min
600/2000 | elapsed: 7.5 min
700/2000 | elapsed: 8.7 min
800/2000 | elapsed: 10.0 min
900/2000 | elapsed: 11.3 min
1000/2000 | elapsed: 12.6 min
1100/2000 | elapsed: 14.0 min
1200/2000 | elapsed: 15.3 min
1300/2000 | elapsed: 16.6 min
1400/2000 | elapsed: 18.0 min
1500/2000 | elapsed: 19.3 min
1600/2000 | elapsed: 20.6 min
1700/2000 | elapsed: 21.9 min
1800/2000 | elapsed: 23.2 min
1900/2000 | elapsed: 24.5 min
2000/2000 | elapsed: 25.8 min


In [57]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_true = benchmark_df["class_label"].tolist()

results_70 = {
    "model": "pruned_70pct",
    "sparsity": 0.70,
    "n_samples": 2000,

    "accuracy": accuracy_score(
        y_true,
        predictions_70
    ),

    "macro_precision": precision_score(
        y_true,
        predictions_70,
        labels=LABELS,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true,
        predictions_70,
        labels=LABELS,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true,
        predictions_70,
        labels=LABELS,
        average="macro",
        zero_division=0
    ),

    "weighted_f1": f1_score(
        y_true,
        predictions_70,
        labels=LABELS,
        average="weighted",
        zero_division=0
    ),

    "total_inference_time_s": total_time_70,

    "samples_per_second":
        2000 / total_time_70,

    "unmatched_count":
        sum(p == "unmatched"
            for p in predictions_70)
}

print("\nEXACT 70% RESULTS")
print("=" * 60)

for k, v in results_70.items():
    print(k, ":", v)


EXACT 70% RESULTS
model : pruned_70pct
sparsity : 0.7
n_samples : 2000
accuracy : 0.0
macro_precision : 0.0
macro_recall : 0.0
macro_f1 : 0.0
weighted_f1 : 0.0
total_inference_time_s : 1548.6616594140005
samples_per_second : 1.2914376667378604
unmatched_count : 2000


In [58]:
PHASE9_DIR = "/kaggle/working/phase9-results"

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned70_2000_results.json"
    ),
    "w"
) as f:
    json.dump(results_70, f, indent=4)

In [59]:
import pandas as pd

pred70_df = benchmark_df.copy()
pred70_df["prediction"] = predictions_70
pred70_df["correct"] = (
    pred70_df["class_label"]
    == pred70_df["prediction"]
)
pred70_df["inference_time_s"] = raw_times_70

pred70_df.to_csv(
    os.path.join(
        PHASE9_DIR,
        "pruned70_2000_predictions.csv"
    ),
    index=False
)

report_70 = classification_report(
    y_true,
    predictions_70,
    labels=LABELS,
    target_names=LABELS,
    zero_division=0
)

with open(
    os.path.join(
        PHASE9_DIR,
        "pruned70_classification_report.txt"
    ),
    "w"
) as f:
    f.write(report_70)

print(report_70)

                                        precision    recall  f1-score   support

                injured_or_dead_people       0.00      0.00      0.00     191.0
rescue_volunteering_or_donation_effort       0.00      0.00      0.00     556.0
                  sympathy_and_support       0.00      0.00      0.00     234.0
     infrastructure_and_utility_damage       0.00      0.00      0.00     214.0
                      not_humanitarian       0.00      0.00      0.00     165.0
                    caution_and_advice       0.00      0.00      0.00     141.0
      displaced_people_and_evacuations       0.00      0.00      0.00     105.0
              requests_or_urgent_needs       0.00      0.00      0.00      68.0
               missing_or_found_people       0.00      0.00      0.00       9.0
            other_relevant_information       0.00      0.00      0.00     317.0

                             micro avg       0.00      0.00      0.00    2000.0
                             macro avg

In [60]:
print("EXACT 70% RESULTS")
print("=" * 60)

for k, v in results_70.items():
    print(k, ":", v)

EXACT 70% RESULTS
model : pruned_70pct
sparsity : 0.7
n_samples : 2000
accuracy : 0.0
macro_precision : 0.0
macro_recall : 0.0
macro_f1 : 0.0
weighted_f1 : 0.0
total_inference_time_s : 1548.6616594140005
samples_per_second : 1.2914376667378604
unmatched_count : 2000


In [61]:
import time
import numpy as np
import torch
import json
import os

LATENCY_TWEET = benchmark_df["text_clean"].iloc[0]

# Warmup
for _ in range(5):
    _ = predict(LATENCY_TWEET, pruned_model, tokenizer)

times_70 = []

for _ in range(50):

    torch.cuda.synchronize()

    start = time.perf_counter()

    _ = predict(
        LATENCY_TWEET,
        pruned_model,
        tokenizer
    )

    torch.cuda.synchronize()

    times_70.append(
        time.perf_counter() - start
    )

latency_70 = {
    "model": "pruned_70pct",
    "warmup_runs": 5,
    "trials": 50,
    "mean_latency_s": float(np.mean(times_70)),
    "median_latency_s": float(np.median(times_70)),
    "std_latency_s": float(np.std(times_70)),
    "min_latency_s": float(np.min(times_70)),
    "max_latency_s": float(np.max(times_70)),
    "throughput_samples_per_s":
        float(1 / np.mean(times_70))
}

print("70% PRUNING — LATENCY RESULTS")
print("=" * 60)

for k, v in latency_70.items():
    print(k, ":", v)

with open(
    os.path.join(PHASE9_DIR, "pruned70_runtime.json"),
    "w"
) as f:
    json.dump(latency_70, f, indent=4)

70% PRUNING — LATENCY RESULTS
model : pruned_70pct
warmup_runs : 5
trials : 50
mean_latency_s : 0.7835662407198469
median_latency_s : 0.7824434344993279
std_latency_s : 0.015123263384311992
min_latency_s : 0.7475376700003835
max_latency_s : 0.8403460620011174
throughput_samples_per_s : 1.276216289105717


In [62]:
import gc

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

_ = predict(
    LATENCY_TWEET,
    pruned_model,
    tokenizer
)

memory_70 = {
    "model": "pruned_70pct",
    "gpus": {}
}

for i in range(torch.cuda.device_count()):

    memory_70["gpus"][f"gpu_{i}"] = {
        "allocated_gb":
            torch.cuda.memory_allocated(i) / 1024**3,

        "reserved_gb":
            torch.cuda.memory_reserved(i) / 1024**3,

        "max_allocated_gb":
            torch.cuda.max_memory_allocated(i) / 1024**3,

        "max_reserved_gb":
            torch.cuda.max_memory_reserved(i) / 1024**3
    }

print("70% PRUNING — GPU MEMORY")
print("=" * 60)

for gpu, vals in memory_70["gpus"].items():
    print("\n", gpu)

    for k, v in vals.items():
        print(f"{k}: {v:.4f}")

with open(
    os.path.join(PHASE9_DIR, "pruned70_memory.json"),
    "w"
) as f:
    json.dump(memory_70, f, indent=4)

70% PRUNING — GPU MEMORY

 gpu_0
allocated_gb: 5.8155
reserved_gb: 5.9082
max_allocated_gb: 5.8273
max_reserved_gb: 5.9082

 gpu_1
allocated_gb: 0.0089
reserved_gb: 2.8730
max_allocated_gb: 0.0089
max_reserved_gb: 2.8730


In [63]:
from codecarbon import EmissionsTracker
import pandas as pd
import time

ENERGY_SAMPLE_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/energy_sample_100.csv"
)

energy_df = pd.read_csv(ENERGY_SAMPLE_PATH)

ENERGY70_DIR = os.path.join(
    PHASE9_DIR,
    "energy_logs",
    "pruned70"
)

os.makedirs(ENERGY70_DIR, exist_ok=True)

tracker = EmissionsTracker(
    output_dir=ENERGY70_DIR,
    output_file="emissions.csv",
    log_level="error"
)

tracker.start()

start = time.perf_counter()

for i, row in energy_df.iterrows():

    _ = predict(
        row["text_clean"],
        pruned_model,
        tokenizer
    )

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/100")

runtime_70_energy = (
    time.perf_counter() - start
)

tracker.stop()

cc70 = pd.read_csv(
    os.path.join(
        ENERGY70_DIR,
        "emissions.csv"
    )
)

last = cc70.iloc[-1]

energy_70 = {
    "model": "pruned_70pct",
    "n_samples": 100,
    "runtime_s": float(runtime_70_energy),
    "energy_kwh":
        float(last["energy_consumed"]),
    "emissions_kg_co2":
        float(last["emissions"])
}

print("\n70% PRUNING — ENERGY RESULTS")
print("=" * 60)

for k, v in energy_70.items():
    print(k, ":", v)

with open(
    os.path.join(PHASE9_DIR, "pruned70_energy.json"),
    "w"
) as f:
    json.dump(energy_70, f, indent=4)

20/100
40/100
60/100
80/100
100/100

70% PRUNING — ENERGY RESULTS
model : pruned_70pct
n_samples : 100
runtime_s : 74.5321868810006
energy_kwh : 0.0024468577306169
emissions_kg_co2 : 0.0011074986426976


In [64]:
# ============================================================
# CREATE PHASE 9 RESULTS BACKUP ZIP
# ============================================================

import os
import shutil

PHASE9_DIR = "/kaggle/working/phase9-results"
BACKUP_DIR = "/kaggle/working/phase9-backup"

# Start clean
if os.path.exists(BACKUP_DIR):
    shutil.rmtree(BACKUP_DIR)

os.makedirs(BACKUP_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Copy all Phase 9 result files
# ------------------------------------------------------------

if os.path.exists(PHASE9_DIR):
    shutil.copytree(
        PHASE9_DIR,
        os.path.join(BACKUP_DIR, "phase9-results")
    )

# ------------------------------------------------------------
# 2. Copy benchmark subset
# ------------------------------------------------------------

benchmark_path = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/benchmark_2000.csv"
)

if os.path.exists(benchmark_path):
    shutil.copy(
        benchmark_path,
        os.path.join(
            BACKUP_DIR,
            "benchmark_2000.csv"
        )
    )

# ------------------------------------------------------------
# 3. Copy Phase 8 metrics for future comparison
# ------------------------------------------------------------

phase8_metrics = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/metrics.csv"
)

if os.path.exists(phase8_metrics):
    shutil.copy(
        phase8_metrics,
        os.path.join(
            BACKUP_DIR,
            "phase8_metrics.csv"
        )
    )

# ------------------------------------------------------------
# 4. Copy energy sample
# ------------------------------------------------------------

energy_sample = (
    "/kaggle/input/datasets/lucky10406/"
    "phase8-final-backup/energy_sample_100.csv"
)

if os.path.exists(energy_sample):
    shutil.copy(
        energy_sample,
        os.path.join(
            BACKUP_DIR,
            "energy_sample_100.csv"
        )
    )

# ------------------------------------------------------------
# 5. Create README
# ------------------------------------------------------------

readme = """
PHASE 9 BACKUP
==============

Project:
LLM compression / Green AI research

Phase 9:
Unstructured magnitude pruning

Pruning levels:
30%
50%
70%

Important:
Each pruning level was created independently from the original
merged FP16 fine-tuned model.

Benchmark:
Fixed stratified 2000-example test subset
Random seed: 42

Energy benchmark:
Fixed 100-example subset used in Phase 8 and Phase 9.

Key Phase 9 findings:

FP16 accuracy:
0.7410

30% pruning:
accuracy = 0.6395
macro F1 = 0.6333022599482043
unmatched = 6 / 2000

50% pruning:
accuracy = 0.0005
macro F1 = 0.0013986013986013986
unmatched = 1992 / 2000

70% pruning:
accuracy = 0.0
macro F1 = 0.0
unmatched = 2000 / 2000

Main finding:
A severe pruning cliff occurs between 30% and 50% sparsity
for one-shot layer-wise unstructured magnitude pruning.

30% dense checkpoint size:
~5.763 GB

Unstructured pruning did not reduce standard dense checkpoint
size because zero-valued weights are still stored.

Important limitation:
50% and 70% pruning were executed primarily on GPU 0,
while some earlier experiments were distributed differently.
Therefore latency/memory comparisons require caution.

DO NOT DELETE:
benchmark_2000.csv
energy_sample_100.csv
Phase 8 metrics
Phase 9 JSON/CSV files

These will be reused during later comparison phases.
"""

with open(
    os.path.join(BACKUP_DIR, "README_PHASE9.txt"),
    "w"
) as f:
    f.write(readme)

# ------------------------------------------------------------
# 6. Create ZIP
# ------------------------------------------------------------

ZIP_BASE = "/kaggle/working/phase9-final-backup"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    BACKUP_DIR
)

print("=" * 70)
print("PHASE 9 BACKUP CREATED")
print("=" * 70)
print("ZIP:")
print(ZIP_BASE + ".zip")

PHASE 9 BACKUP CREATED
ZIP:
/kaggle/working/phase9-final-backup.zip


In [65]:
import zipfile

zip_path = "/kaggle/working/phase9-final-backup.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

print("Files in ZIP:", len(files))

for f in files:
    print(f)

Files in ZIP: 34
phase9-results/
README_PHASE9.txt
benchmark_2000.csv
energy_sample_100.csv
phase8_metrics.csv
phase9-results/energy_logs/
phase9-results/pruned50_energy.json
phase9-results/pruned70_classification_report.txt
phase9-results/pruned70_2000_results.json
phase9-results/pruned70_runtime.json
phase9-results/pruned50_2000_results.json
phase9-results/pruned50_memory.json
phase9-results/pruned30_2000_predictions.csv
phase9-results/pruned30_classification_report.txt
phase9-results/benchmark_2000.csv
phase9-results/pruned50_confusion_matrix.csv
phase9-results/pruned30_2000_results.json
phase9-results/pruned50_runtime.json
phase9-results/metrics_phase8_base.csv
phase9-results/pruned30_confusion_matrix.csv
phase9-results/pruned70_memory.json
phase9-results/pruned50_classification_report.txt
phase9-results/pruned30_memory.json
phase9-results/pruned70_2000_predictions.csv
phase9-results/pruned50_2000_predictions.csv
phase9-results/pruned30_energy.json
phase9-results/pruned70_energy.js

In [66]:
import os

size_mb = os.path.getsize(
    "/kaggle/working/phase9-final-backup.zip"
) / (1024**2)

print(
    "ZIP size:",
    round(size_mb, 2),
    "MB"
)

ZIP size: 0.8 MB
